# Занятие 6. Практика: кастомная трансформация и объединение данных — АВТОРСКОЕ РЕШЕНИЕ

На прошлом занятии мы разобрали теорию: **работа со столбцами и строками**,
пользовательские функции через **`.apply()`** и **`lambda`**, строковый аксессор **`.str`**,
объединение таблиц через **`pd.concat()`** и **`pd.merge()`** (типы соединений `inner`,
`left`, `right`, `outer`).

Сегодня применяем всё это на реальных данных: парсим сырые текстовые поля (ФИО, почта,
телефон), строим новые признаки через `.apply(axis=1)` и собираем финальную таблицу
из основного датасета, второй волны учеников и трёх справочников.

Данные **синтетические** и не соответствуют действительности — телефоны, почты, ФИО, баллы,
города сгенерированы для учебных целей. Совпадения с реальными людьми случайны.

**План занятия:**
1. **Знакомство с данными** — что за пять таблиц лежит в `data/`;
2. **Разминка** — короткие примеры на `.apply`, `.str`, `concat`, `merge`;
3. **Практика** — 12 заданий на 30 баллов.


In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 220)
pd.set_option('display.precision', 2)

---
## Часть 1. Знакомство с данными

Сегодня у нас **пять файлов**:

| Файл | Что это |
|---|---|
| `ege_students.csv`       | основной датасет (то же, что раньше, но с 3 новыми колонками) |
| `ege_students_extra.csv` | «вторая волна» — 50 новых учеников для склейки через `concat` |
| `schools_by_city.csv`    | справочник городов |
| `subject_info.csv`       | справочник школьных предметов |
| `email_domains.csv`      | справочник почтовых доменов |

Загрузим всё и посмотрим по очереди.

In [2]:
df = pd.read_csv('data/ege_students.csv', sep=';')
df_extra = pd.read_csv('data/ege_students_extra.csv', sep=';')
cities = pd.read_csv('data/schools_by_city.csv', sep=';')
subjects = pd.read_csv('data/subject_info.csv', sep=';')
domains = pd.read_csv('data/email_domains.csv', sep=';')

print('df:', df.shape)
print('df_extra:', df_extra.shape)
print('cities:', cities.shape)
print('subjects:', subjects.shape)
print('domains:', domains.shape)

df: (1000, 19)
df_extra: (50, 19)
cities: (8, 4)
subjects: (9, 5)
domains: (6, 3)


### 1.1. Основной датасет — что нового

К уже знакомым колонкам добавились **три «сырых» текстовых поля** из формы регистрации.

In [3]:
df.head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал,ФИО_полное,email,телефон
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да,Смирнов Роман Андреевич,roman_smirnov84@yandex.ru,8 963 557 50 25
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да,Иванова Арина Андреевна,arina2@rambler.ru,8 995 769 12 23
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да,Козлова Дарья Егоровна,darya.kozlova@outlook.com,+7 (952) 136-81-80
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да,Лебедев Владимир Егорович,vladimir_lebedev83@gmail.com,+7-911-910-87-89
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да,Лебедев Дмитрий Николаевич,dmitriy5@yandex.ru,8 965 675 36 93


In [4]:
df.columns

Index(['ученик_id', 'имя', 'пол', 'возраст', 'город', 'тип_школы', 'любимый_предмет', 'часов_подготовки_в_неделю', 'репетитор', 'кол_во_пробников', 'пропусков_уроков', 'мотивация_1_10', 'часов_сна', 'средний_балл_года',
       'балл_егэ', 'сдал', 'ФИО_полное', 'email', 'телефон'],
      dtype='object')

**Новые колонки:**
- `ФИО_полное` — «Фамилия Имя Отчество» одной строкой;
- `email` — почта в формате `логин@домен`;
- `телефон` — номер, но в **разных форматах**.

Посмотрим на них крупным планом:

In [5]:
df[['ФИО_полное', 'email', 'телефон']].head(10)

,ФИО_полное,email,телефон
0,Смирнов Роман Андреевич,roman_smirnov84@yandex.ru,8 963 557 50 25
1,Иванова Арина Андреевна,arina2@rambler.ru,8 995 769 12 23
2,Козлова Дарья Егоровна,darya.kozlova@outlook.com,+7 (952) 136-81-80
3,Лебедев Владимир Егорович,vladimir_lebedev83@gmail.com,+7-911-910-87-89
4,Лебедев Дмитрий Николаевич,dmitriy5@yandex.ru,8 965 675 36 93
5,Кузнецова Милана Сергеевна,milana6@gmail.com,8 943 472 16 38
6,Смирнов Роман Николаевич,roman7@yandex.ru,8 942 673 89 86
7,Голубева Дарья Владимировна,darya.golubeva@mail.ru,+7 (991) 772-56-67
8,Сидоров Тимофей Николаевич,timofey9@yahoo.com,+7 (912) 877-94-38
9,Виноградов Михаил Максимович,mikhail10@outlook.com,+79434639147


### 1.2. Телефоны — разнобой в форматах

Посмотрим на несколько случайных телефонов — как видно, они записаны в разных вариантах.

In [6]:
df['телефон'].sample(15, random_state=0).tolist()

['+79462604766',
 '+7-932-665-59-49',
 '+79876098029',
 '8 996 522 59 40',
 '89948708499',
 '8 970 165 44 92',
 '+79862359332',
 '+79345555685',
 '+7 (901) 313-93-72',
 '+79144151361',
 '+7 (916) 620-23-61',
 '8 995 870 78 39',
 '8 960 782 87 61',
 '89293534666',
 '+7 (932) 265-19-45']

Форматов **пять**: `+7 (999) 123-45-67`, `89991234567`, `+7-999-123-45-67`,
`8 999 123 45 67`, `+79991234567`. В таком виде считать по ним ничего невозможно —
в практике вы напишете свою функцию нормализации.

### 1.3. Домены почт

Посмотрим, какие домены встречаются в колонке `email`:

In [7]:
df['email'].head(10)

0       roman_smirnov84@yandex.ru
1               arina2@rambler.ru
2       darya.kozlova@outlook.com
3    vladimir_lebedev83@gmail.com
4              dmitriy5@yandex.ru
5               milana6@gmail.com
6                roman7@yandex.ru
7          darya.golubeva@mail.ru
8              timofey9@yahoo.com
9           mikhail10@outlook.com
Name: email, dtype: object

### 1.4. `df_extra` — дополнительные ученики

Формат тот же, что и в основном датасете. ID начинаются с **1001**, чтобы не пересекаться.

In [8]:
df_extra.head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал,ФИО_полное,email,телефон
0,1001,Матвей,М,17,Владивосток,Гимназия,Обществознание,18.9,нет,2,11.6,NaN,5.3,3.2,100,да,Васильев Матвей Артёмович,matvey1001@gmail.com,+7 (984) 756-29-83
1,1002,Алиса,Ж,17,Нижний Новгород,Лицей,Математика,16.5,да,11,6.1,4.0,7.8,4.3,95,да,Соловьёва Алиса Владимировна,alisa_soloveva11@gmail.com,+79684425926
2,1003,Софья,Ж,18,Екатеринбург,Гимназия,Обществознание,15.1,да,0,0.4,9.0,6.0,4.2,100,да,Сидорова Софья Владимировна,sofya.sidorova@mail.ru,+7 (933) 742-74-55
3,1004,Артём,М,18,Москва,Лицей,Обществознание,19.7,да,5,4.5,NaN,NaN,2.3,93,да,Зайцев Артём Андреевич,a.zaytsev@mail.ru,8 958 200 28 64
4,1005,Артём,М,16,Ростов-на-Дону,Обычная,Биология,1.6,нет,9,8.4,7.0,7.3,5.0,83,да,Петров Артём Артёмович,artem1005@mail.ru,8 918 617 26 72


In [9]:
# сравним колонки — должны совпадать
print('одинаковые колонки?', df.columns.tolist() == df_extra.columns.tolist())

одинаковые колонки? True


**Важное отличие:** в `df_extra` встречается город **«Владивосток»**, которого нет в основном датасете.

### 1.5. Справочник городов

In [10]:
cities

,город,федеральный_округ,рейтинг_школ,бюджетных_мест_в_вузах
0,Москва,ЦФО,9.2,45000
1,Санкт-Петербург,СЗФО,8.9,28000
2,Казань,ПФО,8.1,12000
3,Новосибирск,СФО,7.8,9500
4,Екатеринбург,УФО,7.5,8800
5,Нижний Новгород,ПФО,7.2,7500
6,Самара,ПФО,6.9,6800
7,Ростов-на-Дону,ЮФО,6.8,7200


**Внимание:** Владивостока здесь **нет**. Значит, после `merge` с этим справочником у учеников из Владивостока `федеральный_округ` и `рейтинг_школ` будут `NaN`.

### 1.6. Справочник предметов

In [11]:
subjects

,любимый_предмет,тип_предмета,средний_балл_по_РФ,максимальный_балл,обязательный
0,Математика,технический,55.6,100,да
1,Русский язык,гуманитарный,68.4,100,да
2,Английский язык,гуманитарный,72.5,100,нет
3,Физика,технический,54.6,100,нет
4,Информатика,технический,59.5,100,нет
5,Обществознание,гуманитарный,58.1,100,нет
6,История,гуманитарный,55.2,100,нет
7,Биология,естественный,51.8,100,нет
8,Химия,естественный,54.3,100,нет


Тут покрыты **все** предметы, которые встречаются в датасете. Пропусков после merge не будет.

### 1.7. Справочник почтовых доменов

In [12]:
domains

,домен_почты,провайдер,страна
0,gmail.com,Google,США
1,yandex.ru,Yandex,Россия
2,mail.ru,VK,Россия
3,yahoo.com,Yahoo,США
4,outlook.com,Microsoft,США
5,rambler.ru,Rambler,Россия


**Внимание:** в датасете есть университетский домен `edu.ru`, а в справочнике его **нет**. Для таких учеников поля `провайдер` и `страна` после merge будут `NaN`.

### 1.8. Итог знакомства

Держите в голове три места, где после объединения появятся `NaN`:

| Справочник | Что не покрыто |
|---|---|
| `cities`   | Владивосток |
| `subjects` | всё покрыто |
| `domains`  | `edu.ru` |

Это НЕ ошибка данных.

---
## Часть 2. Мини-разминка — вспоминаем приёмы

Каждый приём показан **на одном коротком примере, не связанном с задачами практики**.
Дальше приём применяете сами.

### 2.1. `.apply()` с обычной функцией

In [13]:
def age_bucket(x):
    if x < 17:  return 'младше'
    if x == 17: return '17'
    return 'старше'

df['возраст'].apply(age_bucket)

0          17
1          17
2          17
3      младше
4          17
        ...  
995        17
996        17
997        17
998    старше
999        17
Name: возраст, Length: 1000, dtype: object

### 2.2. `lambda` вместо `def`

In [14]:
# признак «сдал ли на 90+» через lambda
df['балл_егэ'].apply(lambda x: x >= 90).sum()

121

### 2.3. `.str.split()` — деление строки

In [15]:
# 'Нижний Новгород' → ['Нижний', 'Новгород']
df['город'].str.split(' ').head()

0    [Нижний, Новгород]
1      [Ростов-на-Дону]
2              [Самара]
3      [Ростов-на-Дону]
4              [Москва]
Name: город, dtype: object

In [16]:
# берём первое слово
df['город'].str.split(' ').apply(lambda x: x[0]).head()

0            Нижний
1    Ростов-на-Дону
2            Самара
3    Ростов-на-Дону
4            Москва
Name: город, dtype: object

### 2.4. `.apply(axis=1)` — функция по всей строке

In [17]:
# часов подготовки на 1 час сна
df.apply(lambda row: row['часов_подготовки_в_неделю'] / (row['часов_сна'] + 1), axis=1).head()

0    0.61
1    1.02
2    1.23
3    1.03
4    0.83
dtype: float64

### 2.5. `pd.concat` — склейка по строкам

In [18]:
a = df.iloc[:3]
b = df.iloc[-3:]
pd.concat([a, b], ignore_index=True)

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал,ФИО_полное,email,телефон
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да,Смирнов Роман Андреевич,roman_smirnov84@yandex.ru,8 963 557 50 25
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да,Иванова Арина Андреевна,arina2@rambler.ru,8 995 769 12 23
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да,Козлова Дарья Егоровна,darya.kozlova@outlook.com,+7 (952) 136-81-80
3,998,Матвей,М,17,Самара,Обычная,История,5.9,нет,4,7,2.0,6.9,2.4,48,да,Семёнов Матвей Александрович,m.semenov@yandex.ru,+7 (943) 393-74-31
4,999,Алиса,Ж,18,Ростов-на-Дону,Обычная,Русский язык,3.1,нет,9,6,8.0,6.2,3.9,74,да,Козлова Алиса Дмитриевна,alisa999@outlook.com,89344816356
5,1000,Алиса,Ж,17,Екатеринбург,Обычная,Физика,14.4,да,5,3,8.0,6.2,4.2,89,да,Иванова Алиса Александровна,alisa1000@yandex.ru,89198717763


### 2.6. `pd.merge` — присоединяем справочник по ключу

In [19]:
mini = pd.DataFrame({
    'тип_школы': ['Лицей', 'Гимназия', 'Обычная'],
    'престиж':   [10, 9, 5],
})
df[['имя', 'тип_школы']].head().merge(mini, on='тип_школы', how='left')

,имя,тип_школы,престиж
0,Роман,Обычная,5
1,Арина,Обычная,5
2,Дарья,Обычная,5
3,Владимир,Обычная,5
4,Дмитрий,Обычная,5


Инструменты вспомнили. **Дальше — практика.**

---
## Практика

Ниже — **12 заданий** на изученный материал. В каждом задании:
- **напишите код** в ячейке `# Ваш код` — код обязателен;
- **выведите результат** через `print()` или последней строкой.

В некоторых заданиях есть пункт **Вывод:** — там нужно короткое предложение о том, что получилось и как это трактовать.

Работаем с таблицами, загруженными выше: основной `df`, вторая волна `df_extra` и три справочника `cities`, `subjects`, `domains`.

**Максимум за практику — 30 баллов.**

### <font color='DarkOrange'>Задание 1 [2 балла]</font>

Из колонки `email` выделите **домен** (часть после `@`) и запишите её в новый столбец `домен_почты`.

Сколько учеников пользуются почтой на `yandex.ru`?

In [20]:
df['домен_почты'] = df['email'].str.split('@').apply(lambda x: x[1])
print((df['домен_почты'] == 'yandex.ru').sum())
# Ответ: 267

267


### <font color='DarkOrange'>Задание 2 [2 балла]</font>

Из колонки `email` выделите **логин** (часть до `@`) и запишите в столбец `логин`.

Сколько логинов состоят **из 8 символов или меньше**?

In [21]:
df['логин'] = df['email'].str.split('@').apply(lambda x: x[0])
print((df['логин'].str.len() <= 8).sum())
# Ответ: 130

130


### <font color='DarkOrange'>Задание 3 [4 балла]</font>

Напишите функцию `normalize_phone(s)`, которая приводит любой формат телефона
(`+7 (999) 123-45-67`, `89991234567`, `+7-999-...`, `8 999 123 45 67`, `+79991234567`)
к единому виду `+7XXXXXXXXXX` (плюс, семёрка, 10 цифр — всего 12 символов).

Идея: оставить только цифры, если первая цифра `8` — заменить на `7`, потом добавить `+` в начало.

Примените функцию к колонке `телефон` и сохраните в `телефон_норм`.

У скольких учеников **нормализованный** телефон **отличается от исходного** — то есть строка в колонке `телефон` не совпадает со строкой в колонке `телефон_норм`?

> Смысл: посчитать, сколько записей реально «пострадали» от нормализации. Те номера, что и так были в формате `+7XXXXXXXXXX` без пробелов и скобок, останутся без изменений — их считать не нужно.

In [22]:
def normalize_phone(s):
    digits = ''.join(ch for ch in s if ch.isdigit())
    if digits.startswith('8'):
        digits = '7' + digits[1:]
    return '+' + digits

df['телефон_норм'] = df['телефон'].apply(normalize_phone)
print((df['телефон'] != df['телефон_норм']).sum())
# Ответ: 798

798


### <font color='DarkOrange'>Задание 4 [2 балла]</font>

По колонке `телефон_норм` возьмите **первые 4 символа** — это «код региона» (пример: `+799`, `+791`).

Сколько учеников имеют телефон, начинающийся с `+799`?


In [23]:
print(df['телефон_норм'].str.startswith('+799').sum())
# Ответ: 120

120


### <font color='DarkOrange'>Задание 5 [2 балла]</font>

Из колонки `ФИО_полное` (формат «Фамилия Имя Отчество») выделите **фамилию** —
первое слово — и сохраните в столбец `фамилия`.

Сколько учеников имеют фамилию, оканчивающуюся на `ов` или `ова`?

> Подсказка: `.str.endswith((...))` умеет принимать кортеж окончаний.

In [24]:
df['фамилия'] = df['ФИО_полное'].str.split(' ').apply(lambda x: x[0])
print(df['фамилия'].str.endswith(('ов', 'ова')).sum())
# Ответ: 712

712


### <font color='DarkOrange'>Задание 6 [3 балла]</font>

Создайте столбец `эффективность` по формуле:

$$
\text{эффективность} = \frac{\text{балл ЕГЭ}}{\text{часов подготовки в неделю} + 1}
$$

(единица в знаменателе — чтобы не делить на ноль). Используйте `.apply(axis=1)`.

Сколько учеников имеют эффективность **строго больше 10**?

In [25]:
df['эффективность'] = df.apply(
    lambda row: row['балл_егэ'] / (row['часов_подготовки_в_неделю'] + 1),
    axis=1
)
print((df['эффективность'] > 10).sum())
# Ответ: 288

288


### <font color='DarkOrange'>Задание 7 [3 балла]</font>

Напишите функцию `categorize(row)`, которая возвращает `'звезда'`, если **одновременно**
`балл_егэ >= 90` и `средний_балл_года >= 4.5`, иначе — `'обычный'`.

Примените через `.apply(axis=1)` и сохраните результат в столбец `статус`.

Сколько получилось «звёзд»?

In [26]:
def categorize(row):
    if row['балл_егэ'] >= 90 and row['средний_балл_года'] >= 4.5:
        return 'звезда'
    return 'обычный'

df['статус'] = df.apply(categorize, axis=1)
print((df['статус'] == 'звезда').sum())
# Ответ: 19

19


### <font color='DarkOrange'>Задание 8 [3 балла]</font>

Склейте `df` и `df_extra` через `pd.concat`, взяв только **исходные 19 колонок**
(тех, что есть в `df_extra` — без ваших `домен_почты`, `логин`, `телефон_норм`,
`фамилия`, `эффективность`, `статус`).

Индексы сбросьте. Результат сохраните в `df_all`.

Сколько в `df_all` учеников из городов, которых **не было** в исходном `df` (то есть эти города «пришли» только со второй волной)?

> Смысл: `concat` расширяет датасет — среди новых строк могут появиться новые категории. Нужно найти множество городов, которые есть в `df_extra`, но отсутствуют в `df`, и посчитать, сколько строк в `df_all` относятся к этим городам.
>
> Подсказка: `set(a) - set(b)` — элементы `a`, которых нет в `b`. Затем `.isin(...)` по колонке `город`.

In [27]:
base_cols = df_extra.columns.tolist()
df_all = pd.concat([df[base_cols], df_extra], ignore_index=True)

new_cities = set(df_extra['город']) - set(df['город'])
print('новые города:', new_cities)
print(df_all['город'].isin(new_cities).sum())
# Ответ: 7

новые города: {'Владивосток'}
7


**Вывод:** со второй волной пришёл только один новый город — Владивосток, и он принёс 7 учеников.

### <font color='DarkOrange'>Задание 9 [2 балла]</font>

Присоедините справочник `cities` к `df_all` через `merge` по колонке `город`.
Тип соединения — такой, чтобы **никого не потерять** из `df_all`.
Результат снова сохраните в `df_all`.

У скольких учеников после merge оказался `NaN` в колонке `федеральный_округ`?

In [28]:
df_all = df_all.merge(cities, on='город', how='left')
print(df_all['федеральный_округ'].isna().sum())
# Ответ: 7

7


### <font color='DarkOrange'>Задание 10 [2 балла]</font>

Из `df_all` возьмите учеников, чей `федеральный_округ == 'ПФО'`.

Какой у них **средний балл ЕГЭ**? Округлите до **двух знаков** после запятой.

In [29]:
sub = df_all[df_all['федеральный_округ'] == 'ПФО']
print(round(sub['балл_егэ'].mean(), 2))
# Ответ: 67.3

67.3


### <font color='DarkOrange'>Задание 11 [2 балла]</font>

Присоедините справочник `subjects` к `df_all` через `merge` по колонке `любимый_предмет`.
Тип соединения — снова такой, чтобы никого не потерять. Результат — в `df_all`.

Сколько в `df_all` учеников с `тип_предмета == 'гуманитарный'`?

In [30]:
df_all = df_all.merge(subjects, on='любимый_предмет', how='left')
print((df_all['тип_предмета'] == 'гуманитарный').sum())
# Ответ: 442

442


### <font color='DarkOrange'>Задание 12 [3 балла]</font>

Сначала создайте в `df_all` столбец `домен_почты` (мы это делали для `df`,
но `df_all` — другая таблица). Потом присоедините справочник `domains` через
`merge` на этот столбец (тип соединения — чтобы никого не потерять).

Сколько учеников удовлетворяют **одновременно** двум условиям:
- `страна == 'Россия'` (из справочника доменов),
- `балл_егэ >= 80`?

In [31]:
df_all['домен_почты'] = df_all['email'].str.split('@').apply(lambda x: x[1])
df_all = df_all.merge(domains, on='домен_почты', how='left')
print(((df_all['страна'] == 'Россия') & (df_all['балл_егэ'] >= 80)).sum())
# Ответ: 176

176


**Вывод:** домен почты работает как прокси-признак «страны провайдера», а ученики на `edu.ru` в подсчёт не попадают, потому что этого домена нет в справочнике и `страна` после `left`-merge у них `NaN`.